In [1]:
import re, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('Netflix_Cleaned.csv')
df.shape        

(8787, 19)

In [2]:
# Step 1: Extract relevant content features

def clean_list(text):
    items = [t.strip() for t in str(text).split(',')]
    # skip "Unknown" so thousands of titles don't look similar just for being unknown
    return [re.sub(r'[^a-z0-9]', '', t.lower()) for t in items if t and t != 'Unknown']

df['genres_tok']   = df['listed_in'].apply(clean_list)
df['director_tok'] = df['director'].apply(clean_list)
df['country_tok']  = df['country'].apply(clean_list)
df['type_tok']     = df['type'].str.lower().str.replace(' ', '')

df[['title', 'genres_tok', 'director_tok', 'country_tok', 'type_tok']].head()

,title,genres_tok,director_tok,country_tok,type_tok
0,Dick Johnson Is Dead,[documentaries],[kirstenjohnson],[unitedstates],movie
1,Ganglands,"[crimetvshows, internationaltvshows, tvactiona...",[julienleclercq],[france],tvshow
2,Midnight Mass,"[tvdramas, tvhorror, tvmysteries]",[mikeflanagan],[unitedstates],tvshow
3,Confessions of an Invisible Girl,"[childrenfamilymovies, comedies]",[brunogarotti],[brazil],movie
4,Sankofa,"[dramas, independentmovies, internationalmovies]",[hailegerima],[unitedstates],movie


Chose genres, director, country and type to describe each title, because the dataset has no description or cast column. I left out ‘Unknown’ values so titles don’t look similar just for missing data

In [3]:
# Step 2: Perform text preprocessing

df['soup'] = df.apply(lambda r: ' '.join(r['genres_tok']*2 + r['director_tok'] + r['country_tok'] + [r['type_tok']]), axis=1)
df['soup'].iloc[1]

'crimetvshows internationaltvshows tvactionadventure crimetvshows internationaltvshows tvactionadventure julienleclercq france tvshow'

Made everything lowercase, removed spaces inside names, and joined each title’s features into one text string. Genres are counted twice so they have the most influence

In [4]:
# Step 3: Calculate content similarity

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['soup'])
X.shape        

(8787, 5114)

Used TF-IDF to turn the text into numbers and cosine similarity to compare titles. The matrix has 8,787 titles and 5,114 features.

In [5]:
# Step 4: Generate recommendations

titles = df['title'].str.lower().reset_index(drop=True)

def recommend(title, n=5):
    idx = titles[titles == title.lower()].index
    if len(idx) == 0:
        return f"'{title}' not found"
    i = idx[0]
    sims = cosine_similarity(X[i], X).ravel()
    sims[i] = -1                       # don't recommend the title itself
    top = np.argsort(-sims)[:n]
    out = df.loc[top, ['title', 'type', 'listed_in', 'director', 'primary_country']].copy()
    out['similarity'] = sims[top].round(3)
    return out

Tested three titles. Midnight Mass returned similar TV horror shows, such as The Haunting of Hill House

In [8]:
recommend('Midnight Mass')

,title,type,listed_in,director,primary_country,similarity
7073,Ratched,TV Show,"TV Dramas, TV Horror, TV Mysteries",Unknown,United States,0.901
7043,The Haunting of Bly Manor,TV Show,"TV Dramas, TV Horror, TV Mysteries",Unknown,United States,0.901
7907,The Haunting of Hill House,TV Show,"TV Dramas, TV Horror, TV Mysteries",Unknown,United States,0.901
6616,Brand New Cherry Flavor,TV Show,"TV Dramas, TV Horror, TV Mysteries",Unknown,United States,0.901
7959,The Originals,TV Show,"TV Dramas, TV Horror, TV Mysteries",Unknown,United States,0.901


In [9]:
recommend('Ganglands')

,title,type,listed_in,director,primary_country,similarity
8405,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Unknown,France,0.863
6735,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Unknown,France,0.863
8331,Agent Raghav,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Unknown,India,0.799
390,The Eagle of El-Se'eed,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Unknown,Pakistan,0.786
486,The Truth,TV Show,"Crime TV Shows, International TV Shows, TV Act...",Unknown,Pakistan,0.786


In [10]:
recommend('Sankofa')

,title,type,listed_in,director,primary_country,similarity
5864,Shanghai,Movie,"Dramas, Independent Movies, International Movies",Dibakar Banerjee,India,0.519
280,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",Dibakar Banerjee,India,0.519
3486,Brain on Fire,Movie,"Dramas, Independent Movies, International Movies",Gerard Barrett,United States,0.514
4675,Chittagong,Movie,"Dramas, Independent Movies, International Movies",Bedabrata Pain,United States,0.514
4736,Dagh Ujala,Movie,"Dramas, Independent Movies, International Movies",Raj Amit Kumar,United States,0.514


In [11]:
# Step 5: Evaluate recommendation quality

rng = np.random.default_rng(42)
sample = rng.choice(len(df), 300, replace=False)
aud = df['audience'].values
hit_model, hit_random = [], []
for i in sample:
    sims = cosine_similarity(X[i], X).ravel(); sims[i] = -1
    top = np.argsort(-sims)[:5]
    hit_model.append((aud[top] == aud[i]).mean())
    hit_random.append((aud[rng.choice(len(df), 5, replace=False)] == aud[i]).mean())
print(f"Same-audience rate: model={np.mean(hit_model):.2f}  random={np.mean(hit_random):.2f}")

Same-audience rate: model=0.50  random=0.32


 To check the recommender, I picked 300 random titles and looked at their top 5 recomendations. For each one I measured how often the recommended titles had the same audience group (Kids, Teens, Adults, etc.) as the original title. I compared this with 5 randomly chosen titles. Audience was not used to build the features, so this is a fair test.

 **Result:** the model matched the audience 50% of the time, against 32% for random picks. This shows the model learns something useful.It is limited because of it only uses same genre,director,country and type, with no plot descriptions.Many titles also tie at the same genre similarity score because they share the same genres.